# 🦅 Kestrel Unified Pretraining Notebook (Kaggle & Local)

Train any of the **4 Kestrel candidate architectures** (`dense`, `hybrid_gdn`, `moe16`, `diff_mhc_relu2`) on **Nemotron-ClimbMix (400B)** streaming straight from Hugging Face.

### Notebook Roadmap
1. **Repo & `uv` Package Install** — Locate/clone repo and install `kestrel[train]` fast via `uv` magic commands.
2. **Secrets & Environment Variables** — Load `HF_TOKEN` & `DAGSHUB_TOKEN` from Kaggle Secrets, Colab Secrets, or local `.env`.
3. **Hardware & Runtime Check** — Inspect GPU(s), VRAM, native BF16/FP16 support, and Muon availability.
4. **Blueprint & Run Config** — Choose architecture, tune overrides, and preview exact total/active parameters.
5. **Pre-Flight Check** — Optional ~15s gradient, mask-leakage, and FP16/BF16 parity check.
6. **Main Training (`kestrel.train`)** — Run on 1 GPU in-process or automatically launch `torchrun` DDP across 2× Kaggle T4s.
7. **Post-Run Curves & Capacity/Plateau Summary** — Inspect loss, `eRank`, scaling slope `alpha`, and throughput offline.

> **Kaggle Setup:** Enable **Internet** + **GPU (T4 ×2 or P100)**, and add `HF_TOKEN` & `DAGSHUB_TOKEN` under **Add-ons → Secrets**.


---
## 1. Repository Setup & `uv` Installation


In [ ]:
# 1.1 Locate local checkout or shallow-clone Kestrel on Kaggle/Colab
# Uses `kestrel_repo` on Kaggle so `/kaggle/working/kestrel` never shadows `kestrel` as a PEP-420 namespace
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/MohammedKamalAlsyd/kestrel.git"
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

REPO_ROOT = None
for candidate in (
    Path.cwd(),
    Path.cwd().parent,
    Path("/kaggle/input/kestrel"),
    WORK_DIR / "kestrel_repo",
    WORK_DIR / "kestrel",
):
    if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "kestrel").is_dir():
        REPO_ROOT = candidate.resolve()
        break

if REPO_ROOT is None:
    clone_dir = WORK_DIR / "kestrel_repo"
    if (clone_dir / ".git").exists():
        subprocess.check_call(["git", "-C", str(clone_dir), "pull", "--ff-only"])
    else:
        if clone_dir.exists() and any(clone_dir.iterdir()):
            shutil.rmtree(clone_dir, ignore_errors=True)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)])
    REPO_ROOT = clone_dir.resolve()
elif (REPO_ROOT / ".git").is_dir() and str(REPO_ROOT).startswith("/kaggle/working"):
    subprocess.call(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"])

print(f"[OK] Kestrel repo: {REPO_ROOT}")


In [ ]:
# 1.2 Switch to repo root and install `kestrel[train]` via `uv` (preserves pre-installed CUDA PyTorch)
%cd {REPO_ROOT}
!command -v uv >/dev/null 2>&1 || pip install -q uv
!uv pip install --system -q -e ".[train]"


In [ ]:
# 1.3 Activate editable install in the running kernel & evict any PEP-420 namespace collision
import importlib
import site
import sys

site.main()
SRC_DIR = str((REPO_ROOT / "src").resolve())
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
os.environ["PYTHONPATH"] = SRC_DIR + (
    os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else ""
)

for mod_name in [m for m in list(sys.modules) if m == "kestrel" or m.startswith("kestrel.")]:
    if getattr(sys.modules[mod_name], "__file__", None) is None:
        del sys.modules[mod_name]
importlib.invalidate_caches()

import kestrel.utils  # noqa: E402

print(f"[OK] kestrel loaded from: {kestrel.__file__}")


---
## 2. Secrets & Environment Variables

Loads secrets without ever printing raw token values (priority: OS env / local `.env` → **Kaggle Secrets** → **Colab Secrets**).


In [ ]:
# 2.1 Reusable secret loaders (local .env, Kaggle Secrets, Colab Secrets)
from kestrel.utils.env import configure_hf_cache, get_work_dir, is_kaggle


def load_dotenv(env_file: Path) -> None:
    """Load KEY=VALUE pairs from a local `.env` file without overwriting existing env vars."""
    if not env_file.is_file():
        return
    for raw_line in env_file.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key, val = key.strip(), val.strip().strip('"').strip("'")
        if key and val and not os.environ.get(key):
            os.environ[key] = val


def load_secret(name: str, default: str | None = None) -> str | None:
    """Resolve a secret from env, Kaggle Secrets, or Colab Secrets and export to `os.environ`."""
    if val := os.environ.get(name):
        return val

    if is_kaggle():
        try:
            from kaggle_secrets import UserSecretsClient  # type: ignore[import-not-found]

            if val := UserSecretsClient().get_secret(name):
                os.environ[name] = val
                return val
        except Exception:
            pass

    try:
        from google.colab import userdata  # type: ignore[import-not-found]

        if val := userdata.get(name):
            os.environ[name] = val
            return val
    except Exception:
        pass

    if default is not None:
        os.environ[name] = default
    return default


In [ ]:
# 2.2 Export secrets, authenticate HF Hub, configure DagsHub MLflow & CUDA allocator
from kestrel.tracking.mlflow_tracker import configure_dagshub_tracking

WORK_DIR = get_work_dir()
HF_CACHE_DIR = configure_hf_cache(WORK_DIR / ".hf_cache")
load_dotenv(REPO_ROOT / ".env")

for secret_key in ("HF_TOKEN", "DAGSHUB_TOKEN", "DAGSHUB_USERNAME", "DAGSHUB_REPO"):
    load_secret(secret_key)

# Explicitly log in to Hugging Face Hub so already-imported `huggingface_hub` / `datasets` use HF_TOKEN
if hf_tok := os.environ.get("HF_TOKEN"):
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_tok
    try:
        from huggingface_hub import login as hf_login

        hf_login(token=hf_tok, add_to_git_credential=False)
    except Exception as exc:
        print(f"[WARN] huggingface_hub.login skipped: {exc}")

# Only configure remote DagsHub MLflow when DAGSHUB_TOKEN is present; otherwise stay cleanly offline SQLite
DAGSHUB_REPO = os.environ.get("DAGSHUB_REPO") or "MohammedKamalAlsyd/kestrel"
MLFLOW_URI = (
    configure_dagshub_tracking(dagshub_repo=DAGSHUB_REPO, token=os.environ.get("DAGSHUB_TOKEN"))
    if os.environ.get("DAGSHUB_TOKEN")
    else None
)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("DATASETS_VERBOSITY", "warning")

print(f"Environment    : {'Kaggle' if is_kaggle() else 'Local'} | Work dir: {WORK_DIR}")
print(f"HF cache       : {HF_CACHE_DIR}")
print(f"MLflow target  : {MLFLOW_URI or 'offline SQLite (no DAGSHUB_TOKEN)'}")
for k in ("HF_TOKEN", "DAGSHUB_TOKEN", "DAGSHUB_USERNAME", "MLFLOW_TRACKING_USERNAME"):
    print(f"  {k:<26s}: {'[SET]' if os.environ.get(k) else '[MISSING]'}")

if is_kaggle() and (not os.environ.get("HF_TOKEN") or not os.environ.get("DAGSHUB_TOKEN")):
    print("\n[TIP] On Kaggle, open Add-ons -> Secrets and ensure the checkbox next to each secret is toggled ON.")


---
## 3. Hardware, Precision & PyTorch Inspection


In [ ]:
# 3.1 Detect GPU(s), VRAM, native BF16/FP16 support, and native Muon availability
import torch

from kestrel.utils.train import get_device_info

HW = get_device_info()
GPU_COUNT = int(HW.get("gpu_count", 0))
VRAM_GB = float(HW.get("vram_gb", 16.0))
HAS_MUON = hasattr(torch.optim, "Muon")

print(f"PyTorch {torch.__version__} | Native Muon: {HAS_MUON}")
for k, v in HW.items():
    print(f"  {k:<24s}: {v}")

if GPU_COUNT == 0:
    print("\n[WARNING] No CUDA GPU detected! On Kaggle, set Session Options -> Accelerator -> GPU T4 x2.")
if not HAS_MUON:
    print("[NOTE] torch.optim.Muon requires PyTorch >= 2.9; falling back to 'adamw'.")


---
## 4. Select Blueprint & Training Configuration

Choose one of the 4 Kestrel candidates (`"dense"`, `"hybrid_gdn"`, `"moe16"`, `"diff_mhc_relu2"`) and adjust steps/batching below.


In [ ]:
# 4.1 User-editable run controls
from kestrel.models import BLUEPRINT_CONFIGS

BLUEPRINT = "dense"  # "dense" | "hybrid_gdn" | "moe16" | "diff_mhc_relu2"
TRAIN_CONFIG_PATH = "configs/train/climbmix_pretrain.json"
MODEL_CONFIG_PATH = BLUEPRINT_CONFIGS[BLUEPRINT]

# Compute & speed
PRECISION = "auto"  # "auto" -> bf16 on Ampere+, fp16+GradScaler on T4, fp32 on CPU
OPTIMIZER_TYPE = "muon_adamw" if HAS_MUON else "adamw"
USE_DDP = GPU_COUNT > 1  # Auto-enables torchrun on Kaggle 2x T4
GRADIENT_CHECKPOINTING = VRAM_GB <= 16.5
COMPILE_MODEL = False  # Set True on Linux/Kaggle for extra Triton fusion speed
LOSS_CHUNK_SIZE = 512  # Chunked lm_head + CE + z-loss (saves ~3-5 GB VRAM)

# Schedule & batching (LOG_INTERVAL logs to MLflow/CSV; PRINT_INTERVAL keeps notebook stdout compact)
MAX_STEPS = 500
MICRO_BATCH_SIZE = 2 if VRAM_GB <= 16.5 else 4
GRAD_ACCUM_STEPS = 16 if MICRO_BATCH_SIZE == 2 else 8
SEQ_LEN = 2048
LOG_INTERVAL = 25
PRINT_INTERVAL = 50
EVAL_INTERVAL = 100
VAL_EVAL_BATCHES = 16
SAVE_INTERVAL = 250
RESUME_FROM = None

# Tracking & Hub publishing (auto-disabled if secrets are missing)
ENABLE_TRACKING = True
USE_DAGSHUB = bool(os.environ.get("DAGSHUB_TOKEN"))
PUSH_TO_HUB = bool(os.environ.get("HF_TOKEN"))
RUN_NAME = None  # None = auto timestamp + blueprint
OUTPUT_DIR = str(WORK_DIR / "artifacts" / "checkpoints")


In [ ]:
# 4.2 Build override list and preview exact total/active parameters on `meta` device (0 VRAM)
from kestrel.models import KestrelModelBuilder

OVERRIDES = [
    f"precision={PRECISION}",
    f"optimizer_type={OPTIMIZER_TYPE}",
    f"gradient_checkpointing={'true' if GRADIENT_CHECKPOINTING else 'false'}",
    f"compile_model={'true' if COMPILE_MODEL else 'false'}",
    f"loss_chunk_size={LOSS_CHUNK_SIZE}",
    f"max_steps={MAX_STEPS}",
    f"micro_batch_size={MICRO_BATCH_SIZE}",
    f"grad_accum_steps={GRAD_ACCUM_STEPS}",
    f"seq_len={SEQ_LEN}",
    f"log_interval={LOG_INTERVAL}",
    f"print_interval={PRINT_INTERVAL}",
    f"eval_interval={EVAL_INTERVAL}",
    f"val_eval_batches={VAL_EVAL_BATCHES}",
    f"save_interval={SAVE_INTERVAL}",
    f"output_dir={OUTPUT_DIR}",
    f"tracking.enabled={'true' if ENABLE_TRACKING else 'false'}",
    f"tracking.push_to_hub={'true' if PUSH_TO_HUB else 'false'}",
]
if USE_DAGSHUB:
    OVERRIDES.append(f"tracking.dagshub_repo={DAGSHUB_REPO}")
else:
    OVERRIDES.append("tracking.dagshub_repo=null")
if RUN_NAME:
    OVERRIDES.append(f"tracking.run_name={RUN_NAME}")
if RESUME_FROM:
    OVERRIDES.append(f"resume_from={RESUME_FROM}")

with torch.device("meta"):
    _meta_model, _ = KestrelModelBuilder.build(MODEL_CONFIG_PATH)
    SUMMARY = KestrelModelBuilder.summarize(_meta_model)

world = GPU_COUNT if USE_DDP else 1
tok_per_step = MICRO_BATCH_SIZE * SEQ_LEN * GRAD_ACCUM_STEPS * max(1, world)
print(f"Blueprint     : {SUMMARY['blueprint']} ({SUMMARY['track']})")
print(f"Parameters    : {SUMMARY['total_params']/1e6:.2f}M total | {SUMMARY['active_params']/1e6:.2f}M active")
print(f"Execution     : {'torchrun DDP x' + str(GPU_COUNT) if USE_DDP else 'single process'}")
print(f"Tokens / step : {tok_per_step:,} | Total ({MAX_STEPS} steps): {tok_per_step * MAX_STEPS:,}")
print(f"Remote sync   : DagsHub={USE_DAGSHUB} | HF Hub push={PUSH_TO_HUB}")


---
## 5. Pre-Flight Gradient, Mask-Leakage & FP16 Parity Check

Runs `kestrel.metrics.preflight` (~10–15s) to verify zero causal/document leakage, healthy Muon/AdamW update ratios, and FP16/BF16 stability before training.


In [ ]:
# 5.1 Run local/Kaggle pre-flight sanity check before committing GPU hours
import json

from kestrel.metrics.preflight import run_preflight_check

RUN_PREFLIGHT = True  # Set False to skip straight to training

if RUN_PREFLIGHT:
    preflight = run_preflight_check(
        MODEL_CONFIG_PATH,
        device="auto",
        full_size=False,
        overfit_steps=25,
        output_dir=WORK_DIR / "experiments" / "preflight",
    )
    print("Pre-flight passed :", preflight["all_passed"])
    print("Leakage check     :", preflight["causal_doc_leakage"])
    print(
        "Overfit loss      :",
        preflight["single_batch_overfit"]["initial_loss"],
        "->",
        preflight["single_batch_overfit"]["final_loss"],
    )
    print("Update ratios     :", preflight["single_batch_overfit"]["update_ratios"])
    print("AMP parity        :", json.dumps(preflight["amp_parity"], indent=2))
    assert preflight["all_passed"], "Pre-flight check failed! Inspect the report above."


---
## 6. Launch Main Training (`kestrel.train`)

- **1 GPU / CPU:** builds `TrainConfig` and calls `kestrel.train.train(cfg)` directly.
- **2+ GPUs (e.g. Kaggle T4 ×2):** launches `torchrun --standalone --nproc_per_node=<N> -m kestrel.train` with live output streaming.


In [ ]:
# 6.1 Prepare resolved TrainConfig (single GPU) or torchrun command (multi-GPU DDP)
import sys

from kestrel.config import TrainConfig, apply_overrides

DDP_CMD = None
TRAIN_CFG = None

if USE_DDP and GPU_COUNT > 1:
    DDP_CMD = [
        sys.executable,
        "-m",
        "torch.distributed.run",
        "--standalone",
        f"--nproc_per_node={GPU_COUNT}",
        "-m",
        "kestrel.train",
        "--train-config",
        TRAIN_CONFIG_PATH,
        "--model-config",
        MODEL_CONFIG_PATH,
    ]
    for ov in OVERRIDES:
        DDP_CMD.extend(["--set", ov])
    print("DDP command ready:\n ", " ".join(DDP_CMD))
else:
    TRAIN_CFG = TrainConfig.from_json(TRAIN_CONFIG_PATH)
    TRAIN_CFG.model_config_path = MODEL_CONFIG_PATH
    TRAIN_CFG = apply_overrides(TRAIN_CFG, OVERRIDES)
    print(
        f"Single-process TrainConfig ready | precision={TRAIN_CFG.precision} "
        f"| steps={TRAIN_CFG.max_steps} | grad_ckpt={TRAIN_CFG.gradient_checkpointing}"
    )


In [ ]:
# 6.2 Execute pretraining
from kestrel.train import train as run_kestrel_train

RUN_RESULT = None

if DDP_CMD is not None:
    proc = subprocess.Popen(
        DDP_CMD,
        cwd=str(REPO_ROOT),
        env=dict(os.environ),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"torchrun exited with status {ret}")
else:
    assert TRAIN_CFG is not None
    RUN_RESULT = run_kestrel_train(TRAIN_CFG)
    print("\nFinal metrics:", json.dumps(RUN_RESULT["metrics"], indent=2))


---
## 7. Post-Run Diagnostics (Loss, Effective Rank `eRank`, Plateau Slope `alpha` & Throughput)

Reads directly from the local `experiments/runs/<run_name>/history.csv` snapshot (works completely offline).


In [ ]:
# 7.1 Load latest run history CSV
import pandas as pd

runs_root = REPO_ROOT / "experiments" / "runs"
snapshot_dir = Path(RUN_RESULT["snapshot_dir"]) if (RUN_RESULT and RUN_RESULT.get("snapshot_dir")) else None

if snapshot_dir is None and runs_root.is_dir():
    run_dirs = sorted(runs_root.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
    snapshot_dir = run_dirs[0] if run_dirs else None

HISTORY_DF = None
if snapshot_dir and (snapshot_dir / "history.csv").is_file():
    HISTORY_DF = pd.read_csv(snapshot_dir / "history.csv")
    print(f"Loaded: {snapshot_dir / 'history.csv'} ({len(HISTORY_DF)} rows)")
    display(HISTORY_DF.tail(5))
else:
    print("No local history.csv found (tracking.enabled may be False).")


In [ ]:
# 7.2 Plot training/validation loss, capacity/plateau signals, and throughput
import matplotlib.pyplot as plt

if HISTORY_DF is not None:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

    if "train/lm" in HISTORY_DF:
        tr = HISTORY_DF.dropna(subset=["train/lm"])
        axes[0].plot(tr["step"], tr["train/lm"], label="train/lm", alpha=0.85)
    if "val/loss" in HISTORY_DF:
        va = HISTORY_DF.dropna(subset=["val/loss"])
        axes[0].plot(va["step"], va["val/loss"], "o-", label="val/loss")
    axes[0].set(title="Language Modeling Loss", xlabel="Step", ylabel="Cross-Entropy")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    plotted_cap = False
    for col in ("capacity/erank", "plateau/delta_per_2x_tokens", "plateau/alpha"):
        if col in HISTORY_DF and HISTORY_DF[col].notna().any():
            sub = HISTORY_DF.dropna(subset=[col])
            axes[1].plot(sub["step"], sub[col], "o-", label=col)
            plotted_cap = True
    axes[1].set(title="Capacity & Plateau Signals", xlabel="Step")
    if plotted_cap:
        axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    if "perf/tokens_per_sec" in HISTORY_DF:
        tp = HISTORY_DF.dropna(subset=["perf/tokens_per_sec"])
        axes[2].plot(tp["step"], tp["perf/tokens_per_sec"], color="tab:green")
    axes[2].set(title="Training Throughput", xlabel="Step", ylabel="Tokens / sec")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
